# Practical 6: Conflation and Single-Pass Clustering

**Aim:** Conflate terms in text documents, show term frequencies, and group similar documents.

Conflation lowercases words, removes stop words, and stems related word forms. TF-IDF represents the processed documents as vectors. Single-pass clustering compares each new document with existing cluster leaders and joins the closest cluster when cosine similarity reaches the threshold.

This notebook reads the same `.txt` files as `practical6.py` from the `data/` folder. It includes five practice news snippets there. For a real-world activity, replace them with four or five article bodies from the [Reuters-21578 collection at UCI](https://archive.ics.uci.edu/dataset/137/reuters+21578+text+categorization+collection).

Install the requirements in a terminal if needed: `python -m pip install nltk scikit-learn`. No NLTK corpus download is needed. Open/run this notebook with its practical folder as the working directory.


In [ ]:
import re
from collections import Counter
from pathlib import Path

from nltk.stem import PorterStemmer
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

threshold = 0.20
stemmer = PorterStemmer()
data_folder = Path("data")
files = sorted(data_folder.glob("*.txt"))
names = [file.name for file in files]
texts = [file.read_text(encoding="utf-8") for file in files]


## 1. Conflate words and count terms

The function keeps alphabetic words, removes common English stop words, then stems the remaining words. The counter shows each document's term-frequency representation.


In [ ]:
def conflate(text):
    words = re.findall(r"[a-zA-Z]+", text.lower())
    return [stemmer.stem(word) for word in words if word not in ENGLISH_STOP_WORDS]

words = [conflate(text) for text in texts]

for name, terms in zip(names, words):
    print(name, dict(Counter(terms)))


## 2. Build TF-IDF vectors

The vectorizer receives the already stemmed words. Cosine similarity will compare document vectors with the current cluster leaders.


In [ ]:
vectorizer = TfidfVectorizer(tokenizer=str.split, token_pattern=None, lowercase=False)
tfidf = vectorizer.fit_transform([" ".join(terms) for terms in words])


## 3. Cluster in one pass

Each document is checked against existing leaders. If its best score is below the threshold, it starts a new cluster. Try changing the threshold or the document order.


In [ ]:
clusters, leaders = [], []
for i in range(len(words)):
    if not leaders:
        leaders.append(i)
        clusters.append([i])
        continue

    scores = cosine_similarity(tfidf[i], tfidf[leaders]).ravel()
    best = scores.argmax()
    if scores[best] >= threshold:
        clusters[best].append(i)
    else:
        leaders.append(i)
        clusters.append([i])

for number, cluster in enumerate(clusters, start=1):
    print(f"\nCluster {number} (leader: {names[leaders[number - 1]]})")
    for i in cluster:
        print(" -", names[i])


## Viva questions with answers

1. **What is conflation in information retrieval?** It groups related word forms under one common form, making similar documents easier to match.
2. **What is a stop word?** A common word, such as “the” or “is,” that is often removed because it carries little search meaning.
3. **What does the Porter stemmer do?** It reduces words to stems, such as “connected” and “connecting” to “connect.”
4. **What does term frequency represent?** How many times each term occurs in a document.
5. **What is TF-IDF?** A weighting method that gives more importance to terms frequent in one document but less common across all documents.
6. **What does cosine similarity measure?** How similar two document vectors are; a higher score means they are more alike.
7. **How does single-pass clustering choose a cluster?** It compares a document with each current cluster leader and joins the closest cluster if the score reaches the threshold.
8. **What is a cluster leader?** The first document assigned to a cluster; later documents are compared with it.
9. **What happens when the similarity threshold increases?** Fewer documents meet the threshold, so more clusters are usually created.
10. **Why can document order change the result?** Each assignment uses the current leaders, and the leaders depend on which documents were processed first.
